# SDXL NPU Conversion
This notebook mounts Google Drive, downloads the Qualcomm QNN SDK, and runs the SDXL conversion pipeline.

In [ ]:
%%bash
# Create 64GB swap space to prevent OOM errors
fallocate -l 64G /swapfile
chmod 600 /swapfile
mkswap /swapfile
swapon /swapfile
free -h


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Empty Google Drive Trash (Optional)
#@markdown Run this cell if you want to permanently empty your Google Drive trash to reclaim space.
#@markdown Note: This will require you to authenticate your Google Account again specifically for the Drive API.

empty_gdrive_trash = False #@param {type:"boolean"}

if empty_gdrive_trash:
    from google.colab import auth
    from googleapiclient.discovery import build
    
    print("Authenticating to access Google Drive API...")
    auth.authenticate_user()
    
    print("Emptying Google Drive trash...")
    drive_service = build("drive", "v3")
    drive_service.files().emptyTrash().execute()
    print("Google Drive trash emptied successfully!")
else:
    print("Skipped emptying trash.")


In [ ]:
import os

# Create working directory
os.makedirs("/content/drive/MyDrive/SDXL", exist_ok=True)
%cd /content/drive/MyDrive/SDXL

!if [ ! -d "convertsdxl" ]; then git clone https://github.com/Lycra-GX/convertsdxl.git --depth=1; fi
%cd convertsdxl

# Download QNN SDK if not exists
!if [ ! -f "qnn_sdk.zip" ]; then wget -q --show-progress "https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip" -O qnn_sdk.zip; fi

# Unzip QNN SDK (extracts to qairt/...)
!if [ ! -d "../qairt" ]; then unzip -q qnn_sdk.zip -d /content/drive/MyDrive/SDXL; fi

# Install uv globally so it is available
!pip install uv

# Setup environment on GDrive and persist it
os.environ["UV_PYTHON_INSTALL_DIR"] = "/content/drive/MyDrive/SDXL/.python"

# Safe check for broken virtualenv interpreter link (avoids os error 39 crash on GDrive)
!if [ -d ".venv" ] && ! .venv/bin/python -c "import sys" 2>/dev/null; then echo "Recreating .venv due to broken symlinks..."; rm -rf .venv; fi
!if [ ! -d ".venv" ]; then uv venv -p 3.10.17; fi
!bash -c "export UV_PYTHON_INSTALL_DIR=/content/drive/MyDrive/SDXL/.python && source .venv/bin/activate && uv sync"


In [ ]:
#@title Setup Export Parameters
model_path = "/content/drive/MyDrive/SDXL/models/anythingxl.safetensors" #@param {type:"string"}
model_name = "anythingxl" #@param {type:"string"}
realistic = False #@param {type:"boolean"}
scheduler = "dpm" #@param ["dpm", "lcm", "eulera"]
cfg = "5,7" #@param {type:"string"}
steps = "15,30" #@param {type:"string"}
soc_version = "min" #@param {type:"string"}

export_script_content = f"""set -e\n\
\n\
model_path=\"{model_path}\"\n\
model_name=\"{model_name}\"\n\
realistic={"true" if realistic else "false"}\n\
scheduler=\"{scheduler}\"\n\
cfg=\"{cfg}\"\n\
steps=\"{steps}\"\n\
\n\
# Define SOC version list\n\
soc_versions=(\"{soc_version}\")\n\
\n\
uv venv -p 3.10.17 --clear\n\
source .venv/bin/activate\n\
uv sync\n\
\n\
# Set realistic flag based on realistic variable\n\
realistic_flag=\"\"\n\
if [ \"$realistic\" = true ]; then\n\
    realistic_flag=\"--realistic\"\n\
fi\n\
\n\
# ======== 1024x1024 ========       \n\
echo \"Processing base resolution: 1024x1024\"\n\
python prepare_data_sdxl.py --model_path $model_path $realistic_flag --scheduler $scheduler --cfg $cfg --step $steps\n\
python gen_quant_data_sdxl.py\n\
python export_onnx_sdxl.py --model_path $model_path\n\
\n\
for soc in \"${{soc_versions[@]}}\"; do\n\
    bash scripts/convert_all_sdxl.sh --min_soc $soc\n\
done\n\
\n\
# ======== Package outputs ========\n\
echo \"Packaging output files...\"\n\
for soc in \"${{soc_versions[@]}}\"; do\n\
    touch output/qnn_models_sdxl_${{soc}}/SDXL\n\
    zip -r ${{model_name}}_qnn2.28_${{soc}}.zip output/qnn_models_sdxl_${{soc}}\n\
done\n\
"""

with open("/content/drive/MyDrive/SDXL/convertsdxl/export_sdxl.sh", "w") as f:
    f.write(export_script_content)

print("export_sdxl.sh successfully generated! Preview:\n")
print(export_script_content)


In [ ]:
%cd /content/drive/MyDrive/SDXL/convertsdxl
!bash export_sdxl.sh